# Waste Classification using Transfer Learning

**What this notebook does, in plain English:**

1. Downloads a small dataset of trash images (6 types: cardboard, glass, metal, paper, plastic, general trash)
2. Trains TWO different AI models to recognize which type of trash is in a photo
3. Compares how well each model did
4. Shows a heatmap on sample images so you can SEE what the model was 'looking at' when it made its decision

**How to use this notebook:**
- This is made for Google Colab (colab.research.google.com)
- Upload this file there, or open it directly
- Go to Runtime -> Change runtime type -> select 'T4 GPU' (this makes training much faster, and it's free)
- Then just click the Play button on each cell, starting from the top, one at a time, and wait for each one to finish before running the next
- Some cells will print progress messages -- that's normal, just wait


## Step 1: Install and import the tools we need

Think of this like gathering your ingredients before cooking. We're just loading libraries (pre-written code) that do the heavy lifting.

In [ ]:
!pip install grad-cam -q

import os
import shutil
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('(This should say cuda -- if it says cpu, go to Runtime > Change runtime type > T4 GPU)')

## Step 2: Download the dataset

We're using **TrashNet**, a small, well-known dataset of ~2,500 trash images split into 6 categories. It's small enough to download and train on quickly.

In [ ]:
# Download the dataset (this is a public dataset, freely available)
!wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip -O dataset.zip

# Unzip it
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('trashnet_raw')

print('Downloaded and extracted. Folders found:')
print(os.listdir('trashnet_raw/dataset-resized'))

## Step 3: Split into training and validation folders

**Why we split the data:** We train the model on most of the images (80%), then test it on images it has NEVER seen before (20%). This tells us how well it would actually perform on new photos, not just ones it memorized.

In [ ]:
source_dir = 'trashnet_raw/dataset-resized'
base_dir = 'trashnet_split'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')

classes = [c for c in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, c))]
print('Classes found:', classes)

random.seed(42)
split_ratio = 0.8  # 80% train, 20% validation

for cls in classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)

    files = os.listdir(os.path.join(source_dir, cls))
    random.shuffle(files)
    split_point = int(len(files) * split_ratio)

    for f in files[:split_point]:
        shutil.copy(os.path.join(source_dir, cls, f), os.path.join(train_dir, cls, f))
    for f in files[split_point:]:
        shutil.copy(os.path.join(source_dir, cls, f), os.path.join(val_dir, cls, f))

print('Done splitting.')
for cls in classes:
    n_train = len(os.listdir(os.path.join(train_dir, cls)))
    n_val = len(os.listdir(os.path.join(val_dir, cls)))
    print(f'{cls}: {n_train} training images, {n_val} validation images')

## Step 4: Prepare the images for the model

AI models need images in a specific size and format. We also add small random changes (flipping, rotating slightly) to the training images ONLY -- this is called **data augmentation** and it helps the model generalize better instead of memorizing exact images.

In [ ]:
img_size = 224  # standard size expected by most pretrained models

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)
print('Class names (in order):', class_names)

## Step 5: A reusable training function

**What is 'transfer learning'?** Instead of teaching a model to recognize images from zero (which needs millions of images), we start with a model that was already trained on 1 million+ general images (ImageNet). It already knows about edges, shapes, textures. We just fine-tune it to recognize OUR 6 trash categories. This is why we can train in minutes instead of days.

In [ ]:
def train_model(model, model_name, num_epochs=8, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        val_loss = val_loss / len(val_loader)
        val_acc = correct / total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'[{model_name}] Epoch {epoch+1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

    return model, history

## Step 6: Build and train Model 1 -- MobileNetV2

**Why MobileNetV2?** It's small and fast -- designed to run on phones/low-power devices. Good choice when you care about deployment, not just raw accuracy.

In [ ]:
mobilenet = models.mobilenet_v2(weights='IMAGENET1K_V1')  # pretrained on ImageNet

# Freeze all the existing layers -- we don't want to erase what it already learned
for param in mobilenet.parameters():
    param.requires_grad = False

# Replace the final classification layer with one matching OUR 6 classes
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, num_classes)

mobilenet, mobilenet_history = train_model(mobilenet, 'MobileNetV2', num_epochs=8)

## Step 7: Build and train Model 2 -- ResNet18

**Why ResNet18?** It's a deeper, generally more accurate architecture -- a good comparison point against the smaller, faster MobileNetV2.

In [ ]:
resnet = models.resnet18(weights='IMAGENET1K_V1')

for param in resnet.parameters():
    param.requires_grad = False

resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)

resnet, resnet_history = train_model(resnet, 'ResNet18', num_epochs=8)

## Step 8: Compare the two models properly

Accuracy alone can be misleading. We also check:
- **Precision**: when the model says 'this is plastic', how often is it actually plastic?
- **Recall**: out of all the actual plastic images, how many did the model correctly catch?
- **F1-score**: a balance between precision and recall

In [ ]:
def evaluate_model(model, model_name):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

    print(f'--- {model_name} ---')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print()
    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name}.png')
    plt.show()

    return {'model': model_name, 'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

mobilenet_results = evaluate_model(mobilenet, 'MobileNetV2')
resnet_results = evaluate_model(resnet, 'ResNet18')

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame([mobilenet_results, resnet_results])
comparison_df = comparison_df.set_index('model')
print('Final comparison table (also save this for your README):')
comparison_df

## Step 9: Grad-CAM -- see what the model is 'looking at'

**In plain terms:** This makes a heatmap on top of an image showing which part of the picture made the model decide 'this is glass' or 'this is paper'. Red/warm areas = the model paid the most attention there. This is the 'explainability' piece -- it shows the model isn't just guessing randomly.

In [ ]:
def show_gradcam(model, model_name, target_layer, num_images=4):
    model.eval()
    cam = GradCAM(model=model, target_layers=[target_layer])

    # grab a few random validation images to visualize
    indices = random.sample(range(len(val_dataset)), num_images)

    fig, axes = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
    fig.suptitle(f'Grad-CAM: {model_name}', fontsize=14)

    for i, idx in enumerate(indices):
        image_tensor, label = val_dataset[idx]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        # undo normalization just for display purposes
        display_img = image_tensor.permute(1, 2, 0).numpy()
        display_img = display_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        display_img = np.clip(display_img, 0, 1)

        grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(label)])[0]
        visualization = show_cam_on_image(display_img, grayscale_cam, use_rgb=True)

        axes[i].imshow(visualization)
        axes[i].set_title(class_names[label])
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(f'gradcam_{model_name}.png')
    plt.show()

# For MobileNetV2, the last conv layer is inside .features
show_gradcam(mobilenet, 'MobileNetV2', target_layer=mobilenet.features[-1])

# For ResNet18, the last conv layer is layer4
show_gradcam(resnet, 'ResNet18', target_layer=resnet.layer4[-1])

## Step 10: Save your trained models (optional, for your GitHub repo)

In [ ]:
torch.save(mobilenet.state_dict(), 'mobilenetv2_waste_classifier.pth')
torch.save(resnet.state_dict(), 'resnet18_waste_classifier.pth')
comparison_df.to_csv('model_comparison_results.csv')

print('Saved: mobilenetv2_waste_classifier.pth, resnet18_waste_classifier.pth, model_comparison_results.csv')
print('Download these from the Colab file browser (folder icon on the left) to add to your GitHub repo.')

## What to put in your GitHub README

- **Problem**: Automated waste classification into 6 categories using image recognition, relevant to IoT-based smart waste sorting systems.
- **Dataset**: TrashNet (~2,500 images, 6 classes).
- **Approach**: Transfer learning using two pretrained architectures -- MobileNetV2 (lightweight, edge-deployable) and ResNet18 (deeper, higher capacity) -- fine-tuned on the classification head.
- **Evaluation**: Accuracy, precision, recall, F1-score, confusion matrices for both models (paste your comparison table here).
- **Explainability**: Grad-CAM visualizations showing model attention regions (attach the saved gradcam images).
- **Next steps you could mention**: converting the smaller model to TensorFlow Lite / ONNX for edge deployment (Raspberry Pi, ESP32-CAM), which you have NOT done but understand is the natural next step.